# Meshes

**Geometry type:** `mesh`

Triangle (or quad) surface meshes: cell boundaries from EM segmentation,
brain surfaces from MRI, organelle hulls.

1. Generate a closed surface (a UV sphere)
2. Write it with a per-vertex attribute
3. Open the store and inspect its metadata
4. Read all geometry and check it against the sphere
5. Bounding-box query
6. Several meshes in one store (`object_ids`)
7. Build a pyramid by edge-collapse decimation
8. OBJ round trip
9. Validate

In [1]:
import os
import tempfile

import numpy as np
import zarr_vectors as zv
from zarr_vectors.types.meshes import read_mesh, write_mesh
from zarr_vectors.validate import validate

_tmpdir = tempfile.mkdtemp(prefix="zv_examples_")
STORE       = os.path.join(_tmpdir, "sphere.zv")
STORE_MULTI = os.path.join(_tmpdir, "cells.zv")
print("Working directory:", _tmpdir)

Working directory: /tmp/zv_examples_7f1ui0dp


## 1 · Generate a closed surface

A UV sphere of radius 50 µm centred at (160, 160, 160) µm, with a synthetic
per-vertex `curvature` near the true 1/R = 0.02 µm⁻¹.

In [2]:
def uv_sphere(radius, lat_steps=40, lon_steps=80, offset=(0.0, 0.0, 0.0)):
    """Vertices and outward-wound triangles of a UV sphere."""
    theta = np.pi * np.arange(lat_steps + 1) / lat_steps
    phi = 2 * np.pi * np.arange(lon_steps) / lon_steps
    t, p = np.meshgrid(theta, phi, indexing="ij")
    verts = np.stack([np.sin(t) * np.cos(p), np.sin(t) * np.sin(p), np.cos(t)], axis=-1)
    verts = (radius * verts.reshape(-1, 3) + offset).astype(np.float32)

    faces = []
    for i in range(lat_steps):
        for j in range(lon_steps):
            a = i * lon_steps + j
            b = a + lon_steps
            c = i * lon_steps + (j + 1) % lon_steps
            d = (i + 1) * lon_steps + (j + 1) % lon_steps
            if i > 0:
                faces.append([a, b, c])
            if i < lat_steps - 1:
                faces.append([b, d, c])
    return verts, np.array(faces, dtype=np.int32)

RADIUS, CENTRE = 50.0, np.array([160.0, 160.0, 160.0])
vertices, faces = uv_sphere(RADIUS, lat_steps=50, lon_steps=100, offset=CENTRE)
curvature = (1 / RADIUS + np.random.default_rng(42).normal(0, 0.001, len(vertices))).astype(np.float32)

print(f"vertices  : {vertices.shape}")
print(f"faces     : {faces.shape}")
print(f"curvature : [{curvature.min():.4f}, {curvature.max():.4f}] µm⁻¹")

vertices  : (5100, 3)
faces     : (9800, 3)
curvature : [0.0164, 0.0235] µm⁻¹


## 2 · Write with a per-vertex attribute

A face whose corners fall in different chunks is stored as a cross-chunk face;
the summary counts both kinds. Vertex attributes may have several columns,
but the viewer stops drawing chunks that carry a multi-column attribute, so
this store keeps a single-column one.

In [3]:
summary = write_mesh(
    STORE,
    vertices=vertices,
    faces=faces,
    chunk_shape=(50.0, 50.0, 50.0),
    bin_shape=(12.5, 12.5, 12.5),
    vertex_attributes={"curvature": curvature},
)
print(summary)

{'vertex_count': 5100, 'face_count': 9800, 'chunk_count': 20, 'intra_face_count': 8530, 'cross_face_count': 1270, 'encoding': 'raw'}


## 3 · Open and inspect

In [4]:
ds = zv.open(STORE)
lv0 = ds.level(0)
print(ds)
print(f"chunk shape  : {lv0.scale} µm")
print(f"objects      : {lv0.objects.count}")
print(f"vertex_count : {lv0.vertex_count:,}")
print(f"vertex attrs : {lv0.attribute_names('vertex')}")
print(f"bounds       : {ds.bounds[0].round(1)} → {ds.bounds[1].round(1)} µm")

Dataset('file:///tmp/zv_examples_7f1ui0dp/sphere.zv', mesh, levels=[0])
chunk shape  : (50.0, 50.0, 50.0) µm
objects      : 1
vertex_count : 5,100
vertex attrs : ('curvature',)
bounds       : [110. 110. 110.] → [210. 210. 210.] µm


## 4 · Read all geometry

`read_mesh` returns every face, including the cross-chunk ones, with `faces`
indexing the returned `vertices` (store order). A UV sphere's flat faces lie
just inside the true sphere, so its area is a little under 4πR².

In [5]:
result = read_mesh(STORE)
v = result["vertices"].astype(np.float64)
f = result["faces"]
print(f"vertex_count : {result['vertex_count']:,}")
print(f"face_count   : {result['face_count']:,}")

cross = np.cross(v[f[:, 1]] - v[f[:, 0]], v[f[:, 2]] - v[f[:, 0]])
area = 0.5 * np.linalg.norm(cross, axis=1).sum()
print(f"surface area : {area:,.1f} µm²  (4πR² = {4 * np.pi * RADIUS ** 2:,.1f} µm²)")

outward = np.einsum("ij,ij->i", cross, v[f].mean(axis=1) - CENTRE) > 0
print(f"outward-wound faces: {outward.sum():,} of {len(f):,}")

vertex_count : 5,100
face_count   : 9,800
surface area : 31,390.1 µm²  (4πR² = 31,415.9 µm²)
outward-wound faces: 9,800 of 9,800


## 5 · Bounding-box query

A bounding-box read returns the faces whose corners all lie inside the box,
and the vertices those faces use.

In [6]:
lo = np.array([110.0, 110.0, 110.0])
hi = np.array([160.0, 160.0, 160.0])

bbox_result = read_mesh(STORE, bbox=(lo, hi))
print(f"faces    : {bbox_result['face_count']:,}")
print(f"vertices : {bbox_result['vertex_count']:,}")

corners_inside = ((vertices[faces] >= lo) & (vertices[faces] <= hi)).all(axis=2)
print(f"input faces with every corner inside: {corners_inside.all(axis=1).sum():,}")

faces    : 1,225
vertices : 750
input faces with every corner inside: 1,225


## 6 · Several meshes in one store

20 small spheres, one object each: `object_ids` gives each vertex its mesh.
`read_mesh` has no object filter; it reads the level, a bounding box or
listed chunks.

In [7]:
rng = np.random.default_rng(5)
verts_list, faces_list = [], []
for i in range(20):
    v_i, f_i = uv_sphere(rng.uniform(8, 20), lat_steps=15, lon_steps=30,
                         offset=rng.uniform(50, 950, 3))
    verts_list.append(v_i)
    faces_list.append(f_i)

offsets = np.cumsum([0] + [len(v_i) for v_i in verts_list[:-1]])
summary = write_mesh(
    STORE_MULTI,
    vertices=np.concatenate(verts_list),
    faces=np.concatenate([f_i + off for f_i, off in zip(faces_list, offsets)]),
    chunk_shape=(100.0, 100.0, 100.0),
    bin_shape=(25.0, 25.0, 25.0),
    object_ids=np.concatenate([np.full(len(v_i), i) for i, v_i in enumerate(verts_list)]),
)
print(summary)
print(f"objects: {zv.open(STORE_MULTI).level(0).objects.count}")

lo_q, hi_q = np.array([200.0, 200.0, 200.0]), np.array([600.0, 600.0, 600.0])
bbox_m = read_mesh(STORE_MULTI, bbox=(lo_q, hi_q))
print(f"faces inside {lo_q} → {hi_q}: {bbox_m['face_count']:,}")

{'vertex_count': 9600, 'face_count': 16800, 'chunk_count': 52, 'intra_face_count': 15503, 'cross_face_count': 1297, 'encoding': 'raw'}
objects: 20


faces inside [200. 200. 200.] → [600. 600. 600.]: 1,680


## 7 · Pyramid by edge-collapse decimation

`method="mesh_decimate"` collapses edges until each level is about 4× smaller
on disk than the one below (coarsen factor 4, sparsity 1), and carries the
per-vertex attributes. It needs scipy (the `[mesh]` extra). The collapsed
vertices sit at their quadric-optimal positions, slightly outside the sphere
on this convex surface, which keeps the area close to the original.

In [8]:
from zarr_vectors_tools.multiresolution.coarsen import build_pyramid

build_pyramid(STORE, factors=[(4, 1), (4, 1)], method="mesh_decimate")

ds = zv.open(STORE)
for lv in ds.levels:
    m = ds.select(level=lv).read()
    pv = m.positions.astype(np.float64)
    tri = np.cross(pv[m.faces[:, 1]] - pv[m.faces[:, 0]], pv[m.faces[:, 2]] - pv[m.faces[:, 0]])
    r = np.linalg.norm(pv - CENTRE, axis=1)
    print(f"Level {lv}: {m.vertex_count:>5,} vertices  {len(m.faces):>5,} faces  "
          f"area {0.5 * np.linalg.norm(tri, axis=1).sum():8,.1f} µm²  "
          f"radius {r.min():.2f}–{r.max():.2f} µm  "
          f"mean curvature {m.attributes['curvature'].mean():.4f}")

Level 0: 5,100 vertices  9,800 faces  area 31,390.1 µm²  radius 50.00–50.00 µm  mean curvature 0.0200
Level 1: 1,227 vertices  2,450 faces  area 31,379.0 µm²  radius 50.00–50.16 µm  mean curvature 0.0200
Level 2:   308 vertices    612 faces  area 31,354.2 µm²  radius 50.01–50.56 µm  mean curvature 0.0201


## 8 · OBJ round trip

In [9]:
from zarr_vectors_tools.convert.export.obj import export_obj
from zarr_vectors_tools.convert.ingest.obj import ingest_obj

obj_in = os.path.join(_tmpdir, "sphere.obj")
with open(obj_in, "w") as fh:
    fh.writelines(f"v {x:.4f} {y:.4f} {z:.4f}\n" for x, y, z in vertices)
    fh.writelines(f"f {a + 1} {b + 1} {c + 1}\n" for a, b, c in faces)   # OBJ is 1-based

store_obj = os.path.join(_tmpdir, "sphere_from_obj.zv")
print(ingest_obj(obj_in, store_obj, (50.0, 50.0, 50.0)))

obj_out = os.path.join(_tmpdir, "sphere_out.obj")
print(export_obj(store_obj, obj_out))
with open(obj_out) as fh:
    lines = fh.readlines()
print(f"exported: {sum(l.startswith('v ') for l in lines):,} vertices, "
      f"{sum(l.startswith('f ') for l in lines):,} faces")

{'vertex_count': 5100, 'face_count': 9800, 'chunk_count': 20, 'intra_face_count': 8530, 'cross_face_count': 1270, 'encoding': 'raw'}
{'vertex_count': 5100, 'face_count': 9800}
exported: 5,100 vertices, 9,800 faces


## 9 · Validate

In [10]:
print(validate(STORE, level=5).summary())
print(validate(STORE_MULTI, level=3).summary())

Level 5 validation: PASS
  52 passed, 0 warnings, 0 errors


Level 3 validation: PASS
  24 passed, 0 warnings, 0 errors


## Summary

| Step | API |
|------|-----|
| Write one mesh | `write_mesh(path, vertices, faces, chunk_shape, bin_shape, vertex_attributes=...)` |
| Write several | `write_mesh(path, vertices, faces, object_ids=...)` |
| Read | `read_mesh(path)` → `vertices`, `faces`, `vertex_count`, `face_count` |
| With attributes | `zarr_vectors.open(path).select(level=n).read()` → `positions`, `faces`, `attributes` |
| Bbox query | `read_mesh(path, bbox=(lo, hi))` (faces with every corner inside) |
| Pyramid | `build_pyramid(path, factors=[(4, 1), (4, 1)], method="mesh_decimate")` |
| OBJ | `ingest_obj(obj_path, store_path, chunk_shape)`, `export_obj(store_path, obj_path)` |

Surface area, normals, curvature, closest point and ray casting are in
`09_mesh_algorithms.ipynb`.